# BOBs — Finding a balanced data split

We have **51 babies and 71 MRI sessions**. Some babies have more than one scan, so we split **participants**, not MRI sessions.

We want around **60% train, 20% validation and 20% test**, with similar age and sex distributions.

## 1. Load the metadata

- We use `participants.csv` from `resampled75`.
- Each row is an MRI session, so some participants appear more than once.

In [15]:
from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

metadata_path = PROJECT_ROOT / "data" / "bobs" / "resampled75" / "participants.csv"
bobs_df = pd.read_csv(metadata_path)

print("Metadata shape:", bobs_df.shape)
print("Columns:", bobs_df.columns.tolist())
print("Participants:", bobs_df["participant_id"].nunique())
print("Sessions:", len(bobs_df))
print("Missing values:", bobs_df.isna().sum().sum())

assert len(bobs_df) == 71
assert bobs_df["participant_id"].nunique() == 51
assert (
    not bobs_df[["participant_id", "session_id", "age", "sex", "pma_weeks"]]
    .isna()
    .any()
    .any()
)

Metadata shape: (71, 7)
Columns: ['participant_id', 'session_id', 'age', 'gestational_age_at_birth', 'sex', 'age_at_scan_weeks', 'pma_weeks']
Participants: 51
Sessions: 71
Missing values: 0


## 2. Look at the ages

- We use the **age at the first available scan** to group each baby.
- We keep all 71 sessions in `bobs_df`; nothing is deleted.
- Our age groups are **1–4, 4–7 and 7–10 months**.

In [16]:
# Keep one row per baby only for deciding the split.
participants_df = (
    bobs_df.sort_values("age").drop_duplicates(subset="participant_id").copy()
)

age_bins = [1, 4, 7, 10]
age_labels = ["1-4", "4-7", "7-10"]

participants_df["age_group"] = pd.cut(
    participants_df["age"], bins=age_bins, labels=age_labels, right=False
)

print("Participants:", len(participants_df))
print("Age groups:")
print(participants_df["age_group"].value_counts().sort_index())
print("Age group vs sex:")
print(pd.crosstab(participants_df["age_group"], participants_df["sex"]))

assert len(participants_df) == 51
assert participants_df["age_group"].notna().all()

Participants: 51
Age groups:
age_group
1-4     22
4-7     21
7-10     8
Name: count, dtype: int64
Age group vs sex:
sex        Female  Male
age_group              
1-4            11    11
4-7            14     7
7-10            6     2


**What we found:** 22 babies are in the 1–4 month group, 21 in 4–7, and 8 in 7–10. The oldest group is smaller, so we need to check that it appears in every split.

## 3. First try — split by age

We first try `seed=42`. It avoids participant overlap, but the sex balance is uneven, and the split sizes are **30 / 10 / 11**.

In [17]:
train_first, remaining_first = train_test_split(
    participants_df,
    test_size=0.4,
    stratify=participants_df["age_group"],
    random_state=42,
)
val_first, test_first = train_test_split(
    remaining_first,
    test_size=0.5,
    stratify=remaining_first["age_group"],
    random_state=42,
)

first_split = {"Train": train_first, "Validation": val_first, "Test": test_first}

for name, subset in first_split.items():
    print(f"{name}: {len(subset)} participants")
    print("Age groups:", subset["age_group"].value_counts().sort_index().to_dict())
    print("Sex:", subset["sex"].value_counts().to_dict())

Train: 30 participants
Age groups: {'1-4': 13, '4-7': 12, '7-10': 5}
Sex: {'Female': 18, 'Male': 12}
Validation: 10 participants
Age groups: {'1-4': 4, '4-7': 4, '7-10': 2}
Sex: {'Female': 8, 'Male': 2}
Test: 11 participants
Age groups: {'1-4': 5, '4-7': 5, '7-10': 1}
Sex: {'Male': 6, 'Female': 5}


## 4. Find a better split

We try several seeds, always splitting at **participant level** and keeping **31 / 10 / 10** babies.

We first look for a similar female/male balance, then compare the ages of **all 71 MRI sessions**. We choose the seed using **metadata only**, not model performance.

In [18]:
# Choose a seed that balances sex and session ages.
target_female = {"Train": 19, "Validation": 6, "Test": 6}
full_groups = pd.cut(bobs_df["age"], bins=age_bins, labels=age_labels, right=False)
full_prop = full_groups.value_counts(normalize=True)

best_score = None
best_seed = None
best_split = None

for seed in range(2000):
    train, remaining = train_test_split(
        participants_df,
        train_size=31,
        stratify=participants_df["age_group"],
        random_state=seed,
    )
    val, test = train_test_split(
        remaining,
        test_size=10,
        stratify=remaining["age_group"],
        random_state=seed,
    )
    splits = {"Train": train, "Validation": val, "Test": test}

    sex_error = sum(
        abs((subset["sex"] == "Female").sum() - target_female[name])
        for name, subset in splits.items()
    )

    session_age_error = 0.0
    for subset in splits.values():
        sessions = bobs_df[bobs_df["participant_id"].isin(subset["participant_id"])]
        session_groups = pd.cut(
            sessions["age"], bins=age_bins, labels=age_labels, right=False
        )
        split_prop = session_groups.value_counts(normalize=True)
        session_age_error += sum(
            abs(split_prop.get(group, 0) - full_prop.get(group, 0))
            for group in age_labels
        )

    score = (int(sex_error), float(session_age_error))
    if best_score is None or score < best_score:
        best_score, best_seed, best_split = score, seed, splits

print("Best seed:", best_seed)
print("Score (sex, session age):", best_score)

for name, subset in best_split.items():
    sessions = bobs_df[bobs_df["participant_id"].isin(subset["participant_id"])]
    session_groups = pd.cut(
        sessions["age"], bins=age_bins, labels=age_labels, right=False
    )
    print(f"{name}: {len(subset)} participants, {len(sessions)} sessions")
    print("Sex:", subset["sex"].value_counts().to_dict())
    print("Session age groups:", session_groups.value_counts().sort_index().to_dict())
    print("Mean age:", round(sessions["age"].mean(), 2))

assert best_seed == 559, "The selected split changed; check metadata and row order."

Best seed: 559
Score (sex, session age): (0, 0.1032863849765257)
Train: 31 participants, 42 sessions
Sex: {'Female': 19, 'Male': 12}
Session age groups: {'1-4': 13, '4-7': 17, '7-10': 12}
Mean age: 5.41
Validation: 10 participants, 15 sessions
Sex: {'Female': 6, 'Male': 4}
Session age groups: {'1-4': 5, '4-7': 6, '7-10': 4}
Mean age: 5.07
Test: 10 participants, 14 sessions
Sex: {'Female': 6, 'Male': 4}
Session age groups: {'1-4': 4, '4-7': 6, '7-10': 4}
Mean age: 5.43


**Selected split: `seed=559`**

| | Train | Validation | Test |
|---|---:|---:|---:|
| Participants | 31 | 10 | 10 |
| MRI sessions | 42 | 15 | 14 |
| Female / Male | 19 / 12 | 6 / 4 | 6 / 4 |
| Mean scan age (months) | 5.41 | 5.07 | 5.43 |

The ages are more evenly spread across the sessions than in the first try.

## 5. Final checks

We check the exact age and PMA ranges, and make sure **every scan belongs to exactly one split**. No baby can be in both train and test.

In [19]:
all_subjects = set()
total_sessions = 0

for name, subset in best_split.items():
    subject_ids = set(subset["participant_id"])
    assert not (all_subjects & subject_ids), "Participant overlap found!"
    all_subjects.update(subject_ids)

    # Put all sessions of each selected baby in the same set.
    sessions = bobs_df[bobs_df["participant_id"].isin(subject_ids)]
    total_sessions += len(sessions)

    print(f"{name}")
    print("Participants:", len(subject_ids))
    print("MRI sessions:", len(sessions))
    print("Age range (months):", sessions["age"].min(), "-", sessions["age"].max())
    print("Mean age:", round(sessions["age"].mean(), 2))
    print(
        "PMA range (weeks):",
        sessions["pma_weeks"].min(),
        "-",
        sessions["pma_weeks"].max(),
    )
    print("Mean PMA:", round(sessions["pma_weeks"].mean(), 2))

assert len(all_subjects) == 51
assert all_subjects == set(bobs_df["participant_id"])
assert total_sessions == len(bobs_df) == 71
print("No participant overlap: OK")
print("All 71 MRI sessions included: OK")

Train
Participants: 31
MRI sessions: 42
Age range (months): 1.7 - 8.9
Mean age: 5.41
PMA range (weeks): 46.8 - 75.5
Mean PMA: 61.14
Validation
Participants: 10
MRI sessions: 15
Age range (months): 1.0 - 8.4
Mean age: 5.07
PMA range (weeks): 46.0 - 72.9
Mean PMA: 60.54
Test
Participants: 10
MRI sessions: 14
Age range (months): 1.8 - 9.4
Mean age: 5.43
PMA range (weeks): 47.9 - 77.7
Mean PMA: 62.04
No participant overlap: OK
All 71 MRI sessions included: OK


## 6. Check the saved JSON

Finally, we compare the IDs from this notebook with the split created by `scripts/create_splits.py`. Both should contain exactly the same babies.

In [20]:
# Run this first from the project root if needed:
# python scripts/create_splits.py --dataset bobs

split_path = PROJECT_ROOT / "data" / "splits" / "bobs_splits.json"
if not split_path.exists():
    raise FileNotFoundError(
        f"Missing {split_path}. Run: python scripts/create_splits.py --dataset bobs"
    )

with open(split_path, encoding="utf-8") as f:
    saved_splits = json.load(f)

for notebook_name, json_name in [
    ("Train", "train"),
    ("Validation", "val"),
    ("Test", "test"),
]:
    notebook_ids = set(best_split[notebook_name]["participant_id"])
    saved_ids = set(saved_splits[json_name])
    assert notebook_ids == saved_ids, f"Different participants in {json_name}!"
    print(f"{json_name}: {len(saved_ids)} participants — identical")

print("All BOBs splits match the notebook!")

train: 31 participants — identical
val: 10 participants — identical
test: 10 participants — identical
All BOBs splits match the notebook!


## Conclusion

We keep **seed=559** for BOBs. The split balances the ages and sex reasonably well, keeps **all 71 scans**, and avoids participant leakage.

The final train/val/test IDs are saved in `data/splits/bobs_splits.json`. We can now use them in our Dataset and DataLoader.